In [7]:
import sys, json, gzip, hmac, hashlib
from pathlib import Path
import numpy as np
import pandas as pd

def find_root(start: Path = Path.cwd()) -> Path:
    for p in [start, *start.parents]:
        if (p / "Makefile").exists() and (p / "configs").exists():
            return p
    return start.parent if start.name in ("notebooks", "notebbook") else start

ROOT = find_root()
for d in ["data/raw", "data/processed", "features", "models", "outputs"]:
    (ROOT / d).mkdir(parents=True, exist_ok=True)

# PERBAIKAN 1: Gunakan jendela 1 Jam (3600 dtk) agar sinyal serangan red-team tidak tenggelam!
WINDOW_SEC = 3600
print("ROOT:", ROOT)

dtypes = {
    "user_hash": "category", "src_computer": "category", "dst_computer": "category",
    "auth_type": "category", "logon_type": "category", "orientation": "category",
    "status": "category", "time": "int64"
}
auth_path = ROOT / "data/raw/lanl_subset_auth.csv.gz"
red_path  = ROOT / "data/raw/lanl_redteam_hashed.csv.gz"

auth = pd.read_csv(auth_path, dtype=dtypes)
red  = pd.read_csv(red_path)

print(f"auth: {auth.shape} | user unik: {auth['user_hash'].nunique()} | redteam events: {len(red)}")

ROOT: c:\Users\ASUS\Downloads\Lomba IBM\trailguard
auth: (15065146, 8) | user unik: 1247 | redteam events: 749


In [8]:
# CELL 2 B2: Ekstraksi Fitur Forensik LANL (Clone Switch + Chain Hop + Foreign Workstation + Relaxed Rarity)
WINDOW_SEC = 3600

def rolling_z(df, col, window=7):
    shifted = df.groupby("user_hash")[col].shift(1)
    grp = shifted.groupby(df["user_hash"])
    mean = grp.rolling(window, min_periods=1).mean().reset_index(level=0, drop=True)
    std = grp.rolling(window, min_periods=2).std().reset_index(level=0, drop=True).fillna(0.0)
    z = (df[col] - mean) / np.maximum(std, 0.25)
    return z.fillna(0.0).clip(-15.0, 15.0)

def build_lanl_features_forensic(auth, red, window_sec=3600):
    print("[INFO] Memproses fitur forensik LANL (Simultaneous Switch, Chain Hop, & Habitual Deviation)...")
    # 1. Urutkan kronologis dan hitung waktu antar-event per user
    auth = auth.sort_values(["user_hash", "time"], kind="stable").reset_index(drop=True)
    auth["time_key"] = (auth["time"] // window_sec).astype("int64")
    auth["user_hash"] = auth["user_hash"].astype(str)
    auth["src_computer"] = auth["src_computer"].astype(str)
    auth["dst_computer"] = auth["dst_computer"].astype(str)

    # Deteksi perpindahan komputer asal secara instan (Simultaneous Clone / Pass-the-Hash) dan Chain Hop (A->B->C)
    prev_src = auth.groupby("user_hash")["src_computer"].shift(1)
    prev_dst = auth.groupby("user_hash")["dst_computer"].shift(1)
    prev_time = auth.groupby("user_hash")["time"].shift(1)
    dt = (auth["time"] - prev_time).fillna(999999.0)

    auth["is_src_switch"] = ((auth["src_computer"] != prev_src) & prev_src.notna()).astype("int8")
    auth["is_rapid_switch_60s"] = ((auth["is_src_switch"] == 1) & (dt <= 60)).astype("int8")
    auth["is_rapid_switch_10s"] = ((auth["is_src_switch"] == 1) & (dt <= 10)).astype("int8")
    auth["is_chain_hop"] = (
        (auth["src_computer"] == prev_dst) &
        (auth["src_computer"] != prev_src) &
        (auth["src_computer"] != auth["dst_computer"]) &
        (dt <= 1800)
    ).astype("int8")

    # 2. Kembalikan urutan berdasarkan waktu global untuk perhitungan kumulatif bebas leakage
    auth = auth.sort_values("time", kind="stable").reset_index(drop=True)
    hour = (auth["time"] % 86400) // 3600
    auth["is_fail"] = (auth["status"] == "Fail").astype("int8")
    auth["is_off"] = ((hour < 7) | (hour >= 19)).astype("int8")
    auth["is_network"] = (auth["logon_type"] == "Network").astype("int8")
    auth["is_kerberos"] = (auth["auth_type"] == "Kerberos").astype("int8")
    auth["is_ntlm"] = (auth["auth_type"].astype(str).str.contains("NTLM", na=False)).astype("int8")
    auth["is_logon"] = (auth["orientation"] == "LogOn").astype("int8")
    auth["is_remote"] = (auth["src_computer"] != auth["dst_computer"]).astype("int8")
    auth["is_rem_logon"] = (auth["is_remote"] & auth["is_logon"]).astype("int8")

    # 3. Frekuensi Historis Kumulatif (Causal: hanya melihat masa lalu)
    auth["u_total_prior"] = auth.groupby("user_hash").cumcount()
    auth["u_src_prior"] = auth.groupby(["user_hash", "src_computer"]).cumcount()
    auth["u_dst_prior"] = auth.groupby(["user_hash", "dst_computer"]).cumcount()
    auth["u_pair_prior"] = auth.groupby(["user_hash", "src_computer", "dst_computer"]).cumcount()

    user_first_tk = auth.groupby("user_hash")["time_key"].transform("min")
    post_cold = (auth["time_key"] > user_first_tk).astype("float64")

    # Pangsa kebiasaan user (seberapa sering user memakai PC asal / tujuan / pasangan ini)
    auth["src_habit_share"] = (auth["u_src_prior"] + 1.0) / (auth["u_total_prior"] + 5.0)
    auth["dst_habit_share"] = (auth["u_dst_prior"] + 1.0) / (auth["u_total_prior"] + 5.0)
    auth["pair_habit_share"] = (auth["u_pair_prior"] + 1.0) / (auth["u_total_prior"] + 5.0)

    # Relaxed Rarity (Tetap aktif menangkap serangan berulang ke-2 s.d. ke-15 di Test Set!)
    auth["is_foreign_src"] = ((auth["src_habit_share"] < 0.20) & (post_cold == 1)).astype("int8")
    auth["is_rare_dst"] = ((auth["u_dst_prior"] <= 12) & (auth["dst_habit_share"] < 0.05) & (post_cold == 1)).astype("int8")
    auth["is_rare_pair"] = ((auth["u_pair_prior"] <= 12) & (auth["pair_habit_share"] < 0.05) & (post_cold == 1)).astype("int8")

    # Kombinasi Sinyal Red-Team pada Event Remote LogOn
    auth["rem_logon_rare_pair"] = (auth["is_rem_logon"] & auth["is_rare_pair"]).astype("int8")
    auth["rem_logon_foreign_src"] = (auth["is_rem_logon"] & auth["is_foreign_src"]).astype("int8")

    # Derajat Kepopuleran Komputer pada Window yang Sama (Membedakan Shared Server vs Target Lateral)
    dst_pop_users = auth.groupby(["time_key", "dst_computer"])["user_hash"].transform("nunique")
    src_pop_dsts = auth.groupby(["time_key", "src_computer"])["dst_computer"].transform("nunique")
    auth["target_exclusivity"] = (1.0 / np.sqrt(dst_pop_users)) * auth["is_rem_logon"]
    auth["pivot_fanout"] = np.log1p(src_pop_dsts) * auth["is_rem_logon"]

    # Skor Risiko Event Gabungan
    auth["event_risk_score"] = (
        post_cold * auth["is_rem_logon"] * (1.0 - auth["pair_habit_share"]) *
        (1.0 + 2.5 * auth["is_rare_pair"] + 2.0 * auth["is_foreign_src"] +
         2.0 * auth["is_rapid_switch_60s"] + 2.5 * auth["is_chain_hop"] +
         1.5 * auth["target_exclusivity"] + 1.0 * auth["is_ntlm"] + 0.5 * auth["is_off"])
    )

    # 4. Agregasi ke level (user_hash, time_key)
    feat = (
        auth.groupby(["user_hash", "time_key"], observed=True)
        .agg(
            total_auth_events=("time", "size"),
            fail_auth_count=("is_fail", "sum"),
            off_hours_event_ratio=("is_off", "mean"),
            unique_src_computers=("src_computer", "nunique"),
            unique_dst_computers=("dst_computer", "nunique"),
            network_logon_ratio=("is_network", "mean"),
            kerberos_auth_ratio=("is_kerberos", "mean"),
            ntlm_auth_ratio=("is_ntlm", "mean"),
            remote_hop_count=("is_remote", "sum"),
            remote_logon_count=("is_rem_logon", "sum"),
            src_switch_count=("is_src_switch", "sum"),
            rapid_switch_60s_count=("is_rapid_switch_60s", "sum"),
            rapid_switch_10s_count=("is_rapid_switch_10s", "sum"),
            chain_hop_count=("is_chain_hop", "sum"),
            foreign_src_count=("is_foreign_src", "sum"),
            rare_dst_count=("is_rare_dst", "sum"),
            rare_pair_count=("is_rare_pair", "sum"),
            rem_logon_rare_pair_count=("rem_logon_rare_pair", "sum"),
            rem_logon_foreign_src_count=("rem_logon_foreign_src", "sum"),
            min_src_habit_share=("src_habit_share", "min"),
            min_dst_habit_share=("dst_habit_share", "min"),
            min_pair_habit_share=("pair_habit_share", "min"),
            max_target_exclusivity=("target_exclusivity", "max"),
            max_pivot_fanout=("pivot_fanout", "max"),
            max_event_risk_score=("event_risk_score", "max"),
            sum_event_risk_score=("event_risk_score", "sum")
        )
        .reset_index()
    )

    # 5. Rasio & Deviasi terhadap Kebiasaan User Sendiri (Mencegah pohon menghafal ID user statis!)
    feat = feat.sort_values(["user_hash", "time_key"]).reset_index(drop=True)
    feat["fail_auth_ratio"] = feat["fail_auth_count"] / (feat["total_auth_events"] + 1.0)
    feat["dst_per_src_ratio"] = feat["unique_dst_computers"] / np.maximum(feat["unique_src_computers"], 1.0)
    feat["rare_pair_ratio"] = feat["rem_logon_rare_pair_count"] / np.maximum(feat["remote_logon_count"], 1.0)
    feat["foreign_src_ratio"] = feat["rem_logon_foreign_src_count"] / np.maximum(feat["remote_logon_count"], 1.0)

    # Ubah rasio protokol statis menjadi deviasi dari rata-rata historis user tersebut
    for r_col in ["kerberos_auth_ratio", "network_logon_ratio", "ntlm_auth_ratio", "off_hours_event_ratio"]:
        u_hist_mean = feat.groupby("user_hash")[r_col].shift(1).groupby(feat["user_hash"]).expanding(min_periods=1).mean().reset_index(level=0, drop=True).fillna(feat[r_col])
        feat[f"{r_col}_dev"] = (feat[r_col] - u_hist_mean).clip(-1.0, 1.0)

    # Rolling Z-Score 7 window
    for col, short in [
        ("total_auth_events", "total_auth"),
        ("unique_dst_computers", "unique_dst"),
        ("unique_src_computers", "unique_src"),
        ("rem_logon_rare_pair_count", "rare_pair"),
        ("rapid_switch_60s_count", "switch60"),
        ("max_event_risk_score", "max_risk")
    ]:
        feat[f"{short}_zscore_7w"] = rolling_z(feat, col, window=7)

    # Z-Score terhadap Populasi pada Jam yang Sama (Stasioner terhadap pergeseran waktu!)
    for col in ["max_event_risk_score", "sum_event_risk_score", "rem_logon_rare_pair_count", "rapid_switch_60s_count", "unique_src_computers"]:
        pop_mean = feat.groupby("time_key")[col].transform("mean")
        pop_std = feat.groupby("time_key")[col].transform("std").fillna(1.0)
        feat[f"{col}_pop_z"] = ((feat[col] - pop_mean) / np.maximum(pop_std, 0.20)).clip(-15.0, 15.0)

    # 6. Pemetaan Label Red-Team
    pos = red.assign(time_key=red["time"] // window_sec)[["user_hash", "time_key"]].drop_duplicates()
    pos["label"] = 1
    feat = feat.merge(pos, on=["user_hash", "time_key"], how="left")
    feat["label"] = feat["label"].fillna(0).astype("int64")

    # 7. Filter Murni Berbasis Fitur (Hanya simpan jam aktif kampanye di mana user melakukan Remote LogOn)
    min_red_tk, max_red_tk = int(pos["time_key"].min()), int(pos["time_key"].max())
    feat = feat[(feat["time_key"] >= max(1, min_red_tk - 2)) & (feat["time_key"] <= max_red_tk + 2)]
    feat = feat[feat["remote_logon_count"] > 0].reset_index(drop=True)

    feat = feat.sort_values(["time_key", "user_hash"]).reset_index(drop=True)
    feature_cols = [c for c in feat.columns if c not in ("time_key", "user_hash", "label")]
    feat[feature_cols] = feat[feature_cols].fillna(0.0).astype("float64")

    print(f"[SUCCESS] Fitur Forensik LANL Selesai! Shape: {feat.shape} | Positif: {int(feat['label'].sum())} | Fitur: {len(feature_cols)}")
    return feat[["time_key", "user_hash", "label"] + feature_cols], feature_cols

# Eksekusi & Simpan ke data/processed/
df_lanl, lanl_feature_cols = build_lanl_features_forensic(auth, red, WINDOW_SEC)
df_lanl.to_csv(ROOT / "data/processed/features_lanl.csv.gz", index=False, compression="gzip")
with open(ROOT / "data/processed/lanl_feature_columns.json", "w") as f:
    json.dump(lanl_feature_cols, f, indent=2)

[INFO] Memproses fitur forensik LANL (Simultaneous Switch, Chain Hop, & Habitual Deviation)...
[SUCCESS] Fitur Forensik LANL Selesai! Shape: (124613, 48) | Positif: 282 | Fitur: 45


In [9]:
def rolling_z(df, col, window=7):
    """Z-score terhadap 7 window aktif sebelumnya milik user yang sama (bebas leakage)."""
    shifted = df.groupby("user_hash")[col].shift(1)
    grp = shifted.groupby(df["user_hash"])
    mean = grp.rolling(window, min_periods=1).mean().reset_index(level=0, drop=True)
    std = grp.rolling(window, min_periods=2).std().reset_index(level=0, drop=True).fillna(0.0)
    # Gunakan pembagi minimal 0.5 agar ketika std=0 nilai z-score tetap proporsional dan tidak meledak
    z = (df[col] - mean) / np.maximum(std, 0.5)
    return z.fillna(0.0).clip(-15.0, 15.0)

def rolling_spike(df, col, window=7):
    """Rasio lonjakan aktivitas saat ini dibanding rata-rata 7 window sebelumnya."""
    shifted = df.groupby("user_hash")[col].shift(1)
    mean = shifted.groupby(df["user_hash"]).rolling(window, min_periods=1).mean().reset_index(level=0, drop=True).fillna(0.0)
    return ((df[col] + 1.0) / (mean + 1.0)).clip(0.0, 50.0)

def add_first_seen(feat, auth, key_cols, name):
    """Jumlah entitas (dst / src / pasangan src-dst) yang PERTAMA KALI diakses user pada window ini."""
    first = auth.groupby(["user_hash"] + key_cols, observed=True)["time_key"].min().reset_index()
    first["user_hash"] = first["user_hash"].astype(str)
    cnt = first.groupby(["user_hash", "time_key"]).size().rename(name).reset_index()
    feat = feat.merge(cnt, on=["user_hash", "time_key"], how="left")
    feat[name] = feat[name].fillna(0.0)
    # Abaikan window pertama kali user muncul (cold-start)
    first_tk = feat.groupby("user_hash")["time_key"].transform("min")
    feat.loc[feat["time_key"] == first_tk, name] = 0.0
    return feat

def build_lanl_features_enhanced(auth, red, window_sec=3600):
    auth = auth.copy()
    auth["time_key"] = (auth["time"] // window_sec).astype("int64")
    hour = (auth["time"] % 86400) // 3600
    
    auth["is_fail"] = (auth["status"] == "Fail").astype("int8")
    auth["is_off"] = ((hour < 7) | (hour >= 19)).astype("int8")
    auth["is_network"] = (auth["logon_type"] == "Network").astype("int8")
    auth["is_kerberos"] = (auth["auth_type"] == "Kerberos").astype("int8")
    auth["is_ntlm"] = (auth["auth_type"].astype(str).str.contains("NTLM", na=False)).astype("int8")

    feat = (
        auth.groupby(["user_hash", "time_key"], observed=True)
        .agg(
            total_auth_events=("time", "size"),
            fail_auth_count=("is_fail", "sum"),
            off_hours_event_ratio=("is_off", "mean"),
            unique_src_computers=("src_computer", "nunique"),
            unique_dst_computers=("dst_computer", "nunique"),
            network_logon_ratio=("is_network", "mean"),
            kerberos_auth_ratio=("is_kerberos", "mean"),
            ntlm_auth_ratio=("is_ntlm", "mean"),
        )
        .reset_index()
    )
    feat["user_hash"] = feat["user_hash"].astype(str)
    feat["fail_auth_ratio"] = feat["fail_auth_count"] / (feat["total_auth_events"] + 1.0)
    feat["dst_per_src_ratio"] = feat["unique_dst_computers"] / np.maximum(feat["unique_src_computers"], 1.0)

    # 1. Fitur Novelty / Lateral Movement (Komputer & Rute Baru)
    feat = add_first_seen(feat, auth, ["dst_computer"], "new_dst_computer_count")
    feat = add_first_seen(feat, auth, ["src_computer"], "new_src_computer_count")
    feat = add_first_seen(feat, auth, ["src_computer", "dst_computer"], "new_pair_count")

    # Rasio entitas baru terhadap aktivitas window ini (Sangat tajam menangkap Red-Team!)
    feat["new_dst_ratio"] = feat["new_dst_computer_count"] / np.maximum(feat["unique_dst_computers"], 1.0)
    feat["new_src_ratio"] = feat["new_src_computer_count"] / np.maximum(feat["unique_src_computers"], 1.0)
    feat["new_pair_ratio"] = feat["new_pair_count"] / np.maximum(feat["total_auth_events"], 1.0)

    # 2. Fitur Self-Baseline (Rolling Z-Score & Spike Ratio 7 Window)
    feat = feat.sort_values(["user_hash", "time_key"]).reset_index(drop=True)
    for col, short in [
        ("total_auth_events", "total_auth"),
        ("unique_dst_computers", "unique_dst"),
        ("unique_src_computers", "unique_src"),
        ("fail_auth_count", "fail_auth"),
        ("new_dst_computer_count", "new_dst"),
        ("new_pair_count", "new_pair")
    ]:
        feat[f"{short}_zscore_7w"] = rolling_z(feat, col, window=7)
        feat[f"{short}_spike_7w"] = rolling_spike(feat, col, window=7)

    # 3. Fitur Cross-Sectional (Deviasi user dibanding rata-rata populasi pada jam yang sama)
    for col in ["unique_dst_computers", "new_pair_count", "total_auth_events"]:
        pop_mean = feat.groupby("time_key")[col].transform("mean")
        pop_std = feat.groupby("time_key")[col].transform("std").fillna(1.0)
        feat[f"{col}_pop_z"] = ((feat[col] - pop_mean) / np.maximum(pop_std, 0.5)).clip(-15.0, 15.0)

    # 4. Pemetaan Label Red-Team
    pos = red.assign(time_key=red["time"] // window_sec)[["user_hash", "time_key"]].drop_duplicates()
    pos["label"] = 1
    feat = feat.merge(pos, on=["user_hash", "time_key"], how="left")
    feat["label"] = feat["label"].fillna(0).astype("int64")
    print(f"Window red-team unik: {len(pos)} | Berhasil dilabeli di tabel fitur: {int(feat['label'].sum())}")

    # Urutkan sesuai Kontrak A
    feat = feat.sort_values(["time_key", "user_hash"]).reset_index(drop=True)
    feature_cols = [c for c in feat.columns if c not in ("time_key", "user_hash", "label")]
    feat[feature_cols] = feat[feature_cols].fillna(0.0).astype("float64")
    return feat[["time_key", "user_hash", "label"] + feature_cols], feature_cols

# Eksekusi & Simpan ke data/processed/
df_lanl, lanl_feature_cols = build_lanl_features_enhanced(auth, red, WINDOW_SEC)
df_lanl.to_csv(ROOT / "data/processed/features_lanl.csv.gz", index=False, compression="gzip")
with open(ROOT / "data/processed/lanl_feature_columns.json", "w") as f:
    json.dump(lanl_feature_cols, f, indent=2)

print(f"[SUCCESS] features_lanl.csv.gz diperbarui! Shape: {df_lanl.shape} | Jumlah Fitur: {len(lanl_feature_cols)}")

Window red-team unik: 282 | Berhasil dilabeli di tabel fitur: 282
[SUCCESS] features_lanl.csv.gz diperbarui! Shape: (196710, 34) | Jumlah Fitur: 31


In [10]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import IsolationForest, HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, f1_score, classification_report, precision_recall_curve

# 1. Potong Temporal pada periode aktif
max_atk_tk = int(df_lanl.loc[df_lanl["label"] == 1, "time_key"].max())
df_active = df_lanl[(df_lanl["time_key"] > 0) & (df_lanl["time_key"] <= max_atk_tk + 24)].reset_index(drop=True)

pos_tk = df_active.loc[df_active["label"] == 1, "time_key"]
t1, t2 = pos_tk.quantile(0.65), pos_tk.quantile(0.82)

tr = df_active[df_active["time_key"] <= t1]
va = df_active[(df_active["time_key"] > t1) & (df_active["time_key"] <= t2)]
te = df_active[df_active["time_key"] > t2]

print(f"Split Evaluasi B2 -> Train: {len(tr):,} (Pos: {tr['label'].sum()}) | Val: {len(va):,} (Pos: {va['label'].sum()}) | Test: {len(te):,} (Pos: {te['label'].sum()})")

# 2. Preprocessing Standar Kontrak B
imp = SimpleImputer(strategy="median").fit(tr[lanl_feature_cols].to_numpy())
sc = RobustScaler().fit(imp.transform(tr[lanl_feature_cols].to_numpy()))

X_tr = sc.transform(imp.transform(tr[lanl_feature_cols].to_numpy()))
X_va = sc.transform(imp.transform(va[lanl_feature_cols].to_numpy()))
X_te = sc.transform(imp.transform(te[lanl_feature_cols].to_numpy()))
y_tr, y_va, y_te = tr["label"].to_numpy(), va["label"].to_numpy(), te["label"].to_numpy()

# 3. Evaluasi Unsupervised (IsolationForest) + Preview Hybrid
iso = IsolationForest(n_estimators=300, max_samples=512, random_state=42, n_jobs=-1).fit(X_tr[y_tr == 0])
unsup_va = -iso.score_samples(X_va)
unsup_te = -iso.score_samples(X_te)

pos_w = float((np.sum(y_tr == 0) / max(np.sum(y_tr == 1), 1)) ** 0.75)
hgb = HistGradientBoostingClassifier(max_depth=4, min_samples_leaf=15, l2_regularization=2.0, class_weight={0: 1.0, 1: pos_w}, random_state=42)
hgb.fit(X_tr, y_tr)

p_va = 0.75 * hgb.predict_proba(X_va)[:, 1] + 0.25 * np.clip((unsup_va - np.percentile(unsup_va, 1)) / (np.percentile(unsup_va, 99) - np.percentile(unsup_va, 1) + 1e-8), 0, 1)
p_te = 0.75 * hgb.predict_proba(X_te)[:, 1] + 0.25 * np.clip((unsup_te - np.percentile(unsup_va, 1)) / (np.percentile(unsup_va, 99) - np.percentile(unsup_va, 1) + 1e-8), 0, 1)

prec, rec, th = precision_recall_curve(y_va, p_va)
f1s = (2 * prec[:-1] * rec[:-1]) / (prec[:-1] + rec[:-1] + 1e-8)
best_th = float(th[np.argmax(f1s)])

print(f"\n=== HASIL EVALUASI FITUR BARU B2 (TEST SET) ===")
print(f"PR-AUC  : {average_precision_score(y_te, p_te):.4f}")
print(f"ROC-AUC : {roc_auc_score(y_te, p_te):.4f}")
print(f"Macro-F1: {f1_score(y_te, (p_te >= best_th).astype(int), average='macro'):.4f}")
print("\nClassification Report (Test Set):")
print(classification_report(y_te, (p_te >= best_th).astype(int), zero_division=0))

Split Evaluasi B2 -> Train: 82,197 (Pos: 187) | Val: 15,567 (Pos: 44) | Test: 93,410 (Pos: 51)

=== HASIL EVALUASI FITUR BARU B2 (TEST SET) ===
PR-AUC  : 0.0232
ROC-AUC : 0.9793
Macro-F1: 0.5298

Classification Report (Test Set):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     93359
           1       0.04      0.16      0.06        51

    accuracy                           1.00     93410
   macro avg       0.52      0.58      0.53     93410
weighted avg       1.00      1.00      1.00     93410

